# The Airline Solver, in Miniature

Two flight constraints. When they point different ways the schedule solver is stable; when they're **near-parallel look-alike layovers** a rounding-error-sized change makes the answer *leap*.

Companion to [`american-airlines-near-parallel-constraints.md`](../docs/american-airlines-near-parallel-constraints.md).

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

def report(name, A, b):
    c1, c2 = A[0], A[1]
    cos = (c1 @ c2) / (np.linalg.norm(c1) * np.linalg.norm(c2))
    ang = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    print(f'--- {name} ---')
    print(f'angle between rows : {ang:8.3f} deg')
    print(f'det(A)             : {np.linalg.det(A):.6e}')
    print(f'cond(A)            : {np.linalg.cond(A):,.1f}')
    x  = np.linalg.solve(A, b)
    x2 = np.linalg.solve(A, b + np.array([0.0, 1e-4]))
    print(f'x                  : {x}')
    print(f'x after 1e-4 nudge : {x2}')
    print(f'answer moved by    : {np.linalg.norm(x2 - x):.4f}')

## Case A — two sensible flights (rows perpendicular)

Rows point different ways → one sharp, stable corner.

In [2]:
A_good = np.array([[1., 0.],
                   [0., 1.]])
b_good = np.array([2., 2.])
report('well-conditioned', A_good, b_good)

--- well-conditioned ---
angle between rows :   90.000 deg
det(A)             : 1.000000e+00
cond(A)            : 1.0
x                  : [2. 2.]
x after 1e-4 nudge : [2.     2.0001]
answer moved by    : 0.0001


## Case B — two look-alike layovers (rows nearly parallel)

Rows differ only in the 4th decimal → a near-duplicate wall. Watch the answer move **14,000×** the size of the input nudge.

In [3]:
A_bad = np.array([[1., 1.],
                  [1., 1.0001]])
b_bad = np.array([2., 2.0001])
report('ill-conditioned', A_bad, b_bad)

--- ill-conditioned ---
angle between rows :    0.003 deg
det(A)             : 1.000000e-04
cond(A)            : 40,002.0
x                  : [1. 1.]
x after 1e-4 nudge : [-0.  2.]
answer moved by    : 1.4142


## The pre-check: catch Case B *before* solving

A near-zero entry on the diagonal of **R** (from `A = QR`) means that row adds almost no new direction — a duplicate.

In [4]:
for name, A in [('CASE A', A_good), ('CASE B', A_bad)]:
    Q, R = np.linalg.qr(A)
    smallest = np.abs(np.diag(R)).min()
    verdict = 'DUPLICATE ROW -- drop/merge!' if smallest < 1e-3 else 'ok'
    print(f'{name}: min |diag(R)| = {smallest:.6f}  ->  {verdict}')

CASE A: min |diag(R)| = 1.000000  ->  ok
CASE B: min |diag(R)| = 0.000071  ->  DUPLICATE ROW -- drop/merge!


### Read it

`angle → 0°`, `det → 0`, `cond → huge` are **three names for the same danger**. A `0.0001` input error became a `1.4` output error. That is exactly what *'the solver went wobbly'* feels like.